# Entity resolution -- Kaggle notebook (branch `features-address-name`), split in 3 roles

Set `MODE` in **cell 4** and run the notebook with **Save & Run All (Commit)**. Internet ON, secret `GITHUB_TOKEN` attached, private dataset with `train/` and `test/` added.

| MODE | What it does | Inputs to attach | Output (Output tab -> New Dataset) | Time (estimate) |
|---|---|---|---|---|
| `"A"` | normalize, validation slice, blocking, features, **train** (val_train) -> `carry/` | the raw dataset | `aml-carry` | 1 - 1.5 h |
| `"COUNTRY"` | streams the test set for the countries in `COUNTRIES` with the saved model -> `out_<country>/` | raw dataset + `aml-carry` | `aml-out-france`, `aml-out-india`, `aml-out-us` | France ~1 h, US ~2.5 h, India ~3.5 h |
| `"MERGE"` | one-to-one + threshold over all countries -> the two TSVs + validator | raw dataset + `aml-carry` + the three `aml-out-*` | `output/matching_results.tsv`, `candidate_pairs.tsv` | 15 - 25 min |

Run A first; then the three COUNTRY notebooks in parallel; then MERGE.

In [ ]:
# Cell 1 -- clone ONLY the branch, then remove the token from .git/config
import os, shutil, subprocess
from kaggle_secrets import UserSecretsClient

GITHUB_REPO = "Payal-mak/amlc26-entity-resolution"      # change if the branch lives in your own fork
BRANCH = "features-address-name"
REPO_DIR = "/tmp/repo"

token = UserSecretsClient().get_secret("GITHUB_TOKEN")
clean_url = f"https://github.com/{GITHUB_REPO}.git"

def git(*args):
    r = subprocess.run(["git", *args], capture_output=True, text=True, encoding="utf-8", errors="replace")
    return r.returncode, (r.stdout + r.stderr).replace(token, "***")

shutil.rmtree(REPO_DIR, ignore_errors=True)
rc, out = git("clone", "-b", BRANCH, "--single-branch", clean_url.replace("https://", f"https://{token}@"), REPO_DIR)
print(out.strip()[-400:])
assert rc == 0, "clone failed -- is the GITHUB_TOKEN secret attached, and Internet ON?"
rc, out = git("-C", REPO_DIR, "remote", "set-url", "origin", clean_url)
assert rc == 0, out
assert token not in open(f"{REPO_DIR}/.git/config").read(), "token still in .git/config!"
print("commit:", git("-C", REPO_DIR, "log", "--oneline", "-1")[1].strip())
COMMIT = git("-C", REPO_DIR, "rev-parse", "HEAD")[1].strip()

In [ ]:
# Cell 2 -- install the pinned requirements
import subprocess, sys
import importlib.metadata as md

REPO_DIR = globals().get("REPO_DIR", "/tmp/repo")
PIPELINE_DIR = f"{REPO_DIR}/student_resource/code/business_entity_resolution"
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{PIPELINE_DIR}/requirements.txt"],
                   capture_output=True, text=True, encoding="utf-8", errors="replace")
noise = ("dependency resolver", "which is incompatible", "requires ", "pip's")
lines = [l for l in (r.stdout + r.stderr).splitlines() if l.strip() and not any(n in l for n in noise)]
print("\n".join(lines[-15:]) or "(no pip output)")
assert r.returncode == 0, "pip install failed"
for pkg in ["duckdb", "pandas", "pyarrow", "lightgbm", "rapidfuzz", "indic-transliteration", "scikit-learn", "psutil"]:
    print(f"  {pkg:24s}{md.version(pkg)}")

In [ ]:
# Cell 3 -- find the dataset folder (the one that contains train/ and test/)
import os

INPUT_ROOT = globals().get("INPUT_ROOT", "/kaggle/input")
NEEDED = ["train/train_source1.tsv", "train/train_source2.tsv", "train/train_source3.tsv", "train/train_ground_truth.tsv",
          "test/test_source1.tsv", "test/test_source2.tsv", "test/test_source3.tsv"]

def find_dirs(root, marker, max_depth=6):
    hits = []
    for dirpath, dirnames, _ in os.walk(root):
        if dirpath[len(root):].count(os.sep) > max_depth:
            dirnames[:] = []
            continue
        if all(os.path.isfile(os.path.join(dirpath, m)) for m in marker):
            hits.append(dirpath)
            dirnames[:] = []
    return hits

hits = find_dirs(INPUT_ROOT, [NEEDED[0], NEEDED[4]])
assert hits, f"no dataset with train/ and test/ under {INPUT_ROOT}: {os.listdir(INPUT_ROOT) if os.path.isdir(INPUT_ROOT) else 'missing'}"
DATA_DIR = hits[0]
missing = [f for f in NEEDED if not os.path.isfile(os.path.join(DATA_DIR, f))]
assert not missing, f"{DATA_DIR} is missing {missing}"
print("DATA_DIR =", DATA_DIR)

In [ ]:
# Cell 4 -- run one role: MODE = "A" (train + carry), "COUNTRY" (stream test for COUNTRIES), or "MERGE"
import os, shutil, subprocess, sys, time

MODE = "A"                       # <-- "A", "COUNTRY" or "MERGE"
COUNTRIES = ["France"]           # COUNTRY mode only: e.g. ["France"], ["India"], ["US"]  (one notebook each, run in parallel)
TEST_CAP = None                  # optional: keep only the top-K candidates per S1 (e.g. 30) to save time
DISABLE_TFIDF = True             # keep the never-tested-at-scale TF-IDF block off (as in the team plan)

REPO_DIR = globals().get("REPO_DIR", "/tmp/repo")
PIPELINE_DIR = f"{REPO_DIR}/student_resource/code/business_entity_resolution"
WORK_DIR = globals().get("WORK_DIR", "/tmp/work")
OUTPUT_DIR = globals().get("OUTPUT_DIR", "/kaggle/working/output")
CARRY_DIR = "/kaggle/working/carry"
INPUT_ROOT = globals().get("INPUT_ROOT", "/kaggle/input")
for d in (WORK_DIR, OUTPUT_DIR):
    os.makedirs(d, exist_ok=True)

CARRY_ITEMS = ["parquet/phase2", "models", "parquet/pipeline/train_report.json", "parquet/phase4/report.json"]

def copy_items(src_root, dst_root):
    for rel in CARRY_ITEMS:
        src, dst = os.path.join(src_root, rel), os.path.join(dst_root, rel)
        if os.path.isdir(src):
            shutil.copytree(src, dst, dirs_exist_ok=True)
        elif os.path.isfile(src):
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            shutil.copy(src, dst)
        else:
            print("WARNING: missing", src)

def restore_carry():
    hits = find_dirs(INPUT_ROOT, ["models/lgbm_folds.pkl", "models/feature_columns.json"])
    assert hits, "attach the aml-carry dataset (output of MODE A) as an input"
    print("carry folder:", hits[0])
    copy_items(hits[0], WORK_DIR)

env = dict(os.environ, AML_DATA_DIR=DATA_DIR, AML_WORK_DIR=WORK_DIR, AML_OUTPUT_DIR=OUTPUT_DIR,
           AML_DUCKDB_MEMORY_LIMIT="8GB",          # the pandas frames + 4 worker processes share the same 30GB
           AML_DUCKDB_THREADS="4", AML_LGBM_THREADS="4", PYTHONUNBUFFERED="1")
common = ["--skip-full-train"] + (["--disable-tfidf-block"] if DISABLE_TFIDF else [])

if MODE == "A":
    runs = [["--stage", s] for s in ["normalize", "val_slice", "val_blocking", "val_features", "val_train"]]
elif MODE == "COUNTRY":
    restore_carry()
    runs = [["--stage", "test_stream", "--countries", ",".join(COUNTRIES)] + (["--test-cap", str(TEST_CAP)] if TEST_CAP else [])]
elif MODE == "MERGE":
    restore_carry()
    import glob
    out_dirs = sorted({os.path.dirname(os.path.dirname(p)) for p in glob.glob(f"{INPUT_ROOT}/**/pred/pred_*.parquet", recursive=True)})
    assert out_dirs, "attach the aml-out-* datasets (outputs of the COUNTRY notebooks) as inputs"
    print("prediction folders:", out_dirs)
    runs = [["--stage", "merge", "--merge-dirs", ",".join(out_dirs)]]
else:
    raise SystemExit("MODE must be A, COUNTRY or MERGE")

t_all = time.time()
log = open(os.path.join(OUTPUT_DIR, f"pipeline_log_{MODE}{'_' + '_'.join(COUNTRIES) if MODE == 'COUNTRY' else ''}.txt"), "a", encoding="utf-8")
for extra in runs:
    cmd = [sys.executable, "-u", "-m", "scripts.run_pipeline", *extra, *common]
    print("$", " ".join(cmd), flush=True)
    p = subprocess.Popen(cmd, cwd=PIPELINE_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, encoding="utf-8", errors="replace", bufsize=1)
    for line in p.stdout:
        print(line, end="")
        log.write(line)
    p.wait()
    log.flush()
    assert p.returncode == 0, f"{extra} failed with exit code {p.returncode}"
log.close()
print(f"\nTOTAL wall time this cell: {(time.time() - t_all) / 3600:.2f} h")

if MODE == "A":
    copy_items(WORK_DIR, CARRY_DIR)
    print("carry folder:", CARRY_DIR, "-> after this run: Output tab -> New Dataset -> name it aml-carry")
elif MODE == "COUNTRY":
    for c in COUNTRIES:
        dst = f"/kaggle/working/out_{c}"
        for sub in ("pred", "cands"):
            os.makedirs(f"{dst}/{sub}", exist_ok=True)
            for f in os.listdir(f"{WORK_DIR}/parquet/stream/{sub}"):
                if f.startswith(f"{sub}_{c}_"):
                    shutil.copy(f"{WORK_DIR}/parquet/stream/{sub}/{f}", f"{dst}/{sub}/{f}")
        print("saved", dst, "-> Output tab -> New Dataset -> name it aml-out-" + c.lower())

In [ ]:
# Cell 5 -- MERGE mode only: check the two submission files
import os, subprocess, sys

REPO_DIR = globals().get("REPO_DIR", "/tmp/repo")
DATA_DIR = globals()["DATA_DIR"]
OUTPUT_DIR = globals().get("OUTPUT_DIR", "/kaggle/working/output")
M, C = f"{OUTPUT_DIR}/matching_results.tsv", f"{OUTPUT_DIR}/candidate_pairs.tsv"
if globals().get("MODE") != "MERGE" or not os.path.exists(M):
    print("nothing to check in this mode (only MODE = 'MERGE' produces the submission files)")
else:
    r = subprocess.run([sys.executable, f"{REPO_DIR}/student_resource/utils/validate_submission.py", "--matching", M,
                        "--candidate", C, "--test-dir", f"{DATA_DIR}/test", "--check-ids"],
                       capture_output=True, text=True, encoding="utf-8", errors="replace")
    print((r.stdout + r.stderr)[-2500:])
    print("VALIDATOR:", "PASS" if r.returncode == 0 else "FAIL", f"(exit code {r.returncode})")
    n_test_s1 = sum(1 for _ in open(f"{DATA_DIR}/test/test_source1.tsv", encoding="utf-8")) - 1
    rows = empty_matches = empty_candidates = n_matched_ids = not_in_candidates = 0
    examples = []
    with open(M, encoding="utf-8") as fm, open(C, encoding="utf-8") as fc:
        assert fm.readline().rstrip("\n").split("\t") == ["source1_entity_id", "matched_entity_ids"]
        assert fc.readline().rstrip("\n").split("\t") == ["source1_entity_id", "candidate_entity_ids"]
        for lm, lc in zip(fm, fc):
            s1m, _, matched = lm.rstrip("\n").partition("\t")
            s1c, _, cands = lc.rstrip("\n").partition("\t")
            assert s1m == s1c, f"files are not in the same S1 order: {s1m} vs {s1c}"
            rows += 1
            empty_matches += not matched
            empty_candidates += not cands
            if matched:
                cset = set(cands.split(","))
                for m_id in matched.split(","):
                    n_matched_ids += 1
                    if m_id not in cset:
                        not_in_candidates += 1
                        if len(examples) < 5:
                            examples.append((s1m, m_id))
    print(f"test S1 expected: {n_test_s1:,}   rows: {rows:,}")
    print(f"S1 with EMPTY matches: {empty_matches:,} ({empty_matches / max(rows, 1):.1%})   S1 with empty candidates: {empty_candidates:,}")
    print(f"matched ids: {n_matched_ids:,}   NOT in candidate_pairs.tsv: {not_in_candidates:,}", "-> OK" if not_in_candidates == 0 else f"-> PROBLEM {examples}")
    for f in (M, C):
        print(f"  {os.path.basename(f)}: {os.path.getsize(f) / 1e6:.0f} MB")
    print("\nOVERALL:", "PASS" if (r.returncode == 0 and rows == n_test_s1 and not_in_candidates == 0) else "FAIL")

In [ ]:
# Cell 6 -- keep the validation report with the outputs (MODE A and MERGE)
import json, os

WORK_DIR = globals().get("WORK_DIR", "/tmp/work")
OUTPUT_DIR = globals().get("OUTPUT_DIR", "/kaggle/working/output")
COMMIT = globals().get("COMMIT", "unknown")
val = {}
p = f"{WORK_DIR}/parquet/phase4/report.json"
if os.path.isfile(p):
    val = json.load(open(p, encoding="utf-8"))
by_group = val.get("by_group", {})
summary = {
    "mode": globals().get("MODE"), "commit": COMMIT, "branch": "features-address-name",
    "validation_slice_oof": {
        "macro_f0.5": val.get("macro_f_beta"), "micro_precision": val.get("micro_precision"),
        "micro_recall": val.get("micro_recall"), "singleton_accuracy": val.get("singleton_accuracy"),
        "policy": val.get("policy"), "India_f0.5": (by_group.get("India") or {}).get("f_beta"),
        "US_f0.5": (by_group.get("US") or {}).get("f_beta"), "n_entities": val.get("n_entities"),
    },
}
os.makedirs(OUTPUT_DIR, exist_ok=True)
json.dump(summary, open(f"{OUTPUT_DIR}/report_{summary['mode']}.json", "w", encoding="utf-8"), indent=2, default=str)
print(json.dumps(summary, indent=2, default=str))